In [1]:
import numpy as np, glob
from astropy.io import fits
from scipy.optimize import curve_fit
import fits_reprocess as fr

files = sorted(glob.glob(r'E:/Reverse Telescope Test Data/20260213_data/allmetal/allmetal_fits/*.fits'))
print('n files', len(files))

def gaussian(x, amp, mu, sigma, offset):
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2) + offset

FWHM_FACTOR = 2.0 * np.sqrt(2.0 * np.log(2.0))

n files 3922


In [2]:
def _estimate_sigma(profile, x=None, mu_guess=None):
    """Width of the above-background run containing the peak, converted to sigma.

    Same median cut as before. The fix is that only the contiguous run around the
    max is counted, so rectified noise elsewhere in the profile carries no weight.
    """
    profile = np.asarray(profile, dtype=np.float64)
    shifted = np.clip(profile - np.median(profile), 0.0, None)
    i0 = int(shifted.argmax())
    li = i0
    while li > 0 and shifted[li - 1] > 0:
        li -= 1
    ri = i0
    while ri < shifted.size - 1 and shifted[ri + 1] > 0:
        ri += 1
    return (ri - li + 1) / FWHM_FACTOR

In [3]:
def _fit_one_profile(profile):
    """Fit a Gaussian to a 1D profile. Returns (amp, mu, sigma, offset) or NaNs.

    Tries the estimate and a factor of 2 either side, keeps the lowest residual.
    """
    profile = np.asarray(profile, dtype=np.float64)
    n = profile.size
    x = np.arange(n, dtype=np.float64)
    baseline = float(np.median(profile))
    mu_guess = float(profile.argmax())
    amp_guess = float(profile.max()) - baseline      # amp is height ABOVE offset
    sigma_est = _estimate_sigma(profile)

    sig_hi = n / 2.0
    bounds = ([0.0, 0.0, 1.0, -np.inf], [np.inf, float(n), sig_hi, np.inf])

    best, best_resid = None, np.inf
    for s in [sigma_est, sigma_est * 2.0, sigma_est * 0.5]:
        s = float(np.clip(s, 1.001, sig_hi * 0.999))
        try:
            popt, _ = curve_fit(gaussian, x, profile,
                                p0=[amp_guess, mu_guess, s, baseline],
                                bounds=bounds, maxfev=5000)
        except Exception:
            continue
        resid = float(np.sqrt(np.mean((gaussian(x, *popt) - profile) ** 2)))
        if resid < best_resid:
            best, best_resid = popt, resid

    if best is None:
        return np.nan, np.nan, np.nan, np.nan
    amp, mu, sigma, offset = best
    return amp, mu, abs(sigma), offset

In [4]:
for idx in [0, 10, 17]:      # frame 1 (good), 11 and 18 (railed under the old code)
    with fits.open(files[idx]) as h:
        img = np.flip(h[0].data, axis=(0, 1)).astype(float)
    py = np.sum(img, axis=1)
    y = np.arange(py.size)
    mu_g = float(py.argmax())

    print()
    print('=== frame %d' % (idx + 1))
    print('  OLD sigma_est %8.2f   NEW sigma_est %8.2f'
          % (fr._estimate_sigma(py, y, mu_g), _estimate_sigma(py)))
    print('  OLD fit  -> mu=%10.4f  sigma=%9.4f' % fr._fit_one_profile(py)[1:3])
    print('  NEW fit  -> mu=%10.4f  sigma=%9.4f' % _fit_one_profile(py)[1:3])
    t, _ = curve_fit(gaussian, y, py, p0=[py.max(), py.argmax(), 5, np.median(py)])
    print('  TRUTH    -> mu=%10.4f  sigma=%9.4f' % (t[1], t[2]))


=== frame 1
  OLD sigma_est   256.00   NEW sigma_est    12.32
  OLD fit  -> mu=  544.6194  sigma=   5.0912
  NEW fit  -> mu=  544.6194  sigma=   5.0911
  TRUTH    -> mu=  544.6196  sigma=   5.0908

=== frame 11
  OLD sigma_est   256.00   NEW sigma_est     8.92
  OLD fit  -> mu=    0.0000  sigma= 355.2611
  NEW fit  -> mu=  535.2671  sigma=   4.4060
  TRUTH    -> mu=  535.2666  sigma=   4.4065

=== frame 18
  OLD sigma_est   251.97   NEW sigma_est    11.47
  OLD fit  -> mu=  410.3647  sigma= 512.0000
  NEW fit  -> mu=  530.6107  sigma=   5.3537
  TRUTH    -> mu=  530.6110  sigma=   5.3534
